# 07 - Validacion estadistica

El README tira tres numeros gruesos: AOV alrededor de $86, retencion M1 creciente, y un triangulo de cohortes que sugiere mejora reciente. Esta notebook los audita con herramientas que no son SQL:

1. **Intervalo de confianza del AOV** via bootstrap no parametrico (1000 replicas). Responde si el AOV del ultimo mes cerrado es estable o si lo mueve un outlier.
2. **Test de significancia sobre retencion M1** comparando cohortes maduras de la primera mitad vs la segunda mitad del periodo. Responde si el rebote de retencion que se ve en el triangulo es real o ruido de muestreo.
3. **Heatmap de retencion** reconstruido en Python con cohortes y celdas observables hasta el ultimo mes cerrado, para no mezclar meses parciales con comportamiento real.

La data se lee directamente desde BigQuery (`bigquery-public-data.thelook_ecommerce`) y se fija un `last_complete_month`: el mes anterior al maximo disponible del dataset. Esto evita reportar abril parcial como si fuera mes cerrado.

> **Nota de reproducibilidad:** para correr la notebook hace falta Application Default Credentials de GCP (`gcloud auth application-default login`) y un proyecto con facturacion. Cada query procesa menos de 1 GB, entra en el free tier.


## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

project_root = Path.cwd()
if not (project_root / "sql_queries").exists():
    project_root = project_root.parent

DASHBOARD_DIR = project_root / "dashboards"
DASHBOARD_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# Cliente de BigQuery. Usa ADC; el proyecto de facturacion se infiere del env.
client = bigquery.Client()

REFERENCE_QUERY = """
SELECT
  MAX(DATE(created_at)) AS max_order_date,
  DATE_TRUNC(MAX(DATE(created_at)), MONTH) AS max_order_month,
  DATE_TRUNC(DATE_SUB(MAX(DATE(created_at)), INTERVAL 1 MONTH), MONTH) AS last_complete_month
FROM `bigquery-public-data.thelook_ecommerce.order_items`
WHERE status NOT IN ('Cancelled', 'Returned')
"""

reference = client.query(REFERENCE_QUERY).to_dataframe(create_bqstorage_client=False).iloc[0]
max_order_date = pd.Timestamp(reference["max_order_date"])
last_complete_month = pd.Timestamp(reference["last_complete_month"])
query_params = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("last_complete_month", "DATE", last_complete_month.date())
    ]
)

print(f"Proyecto de facturacion: {client.project}")
print(f"Maxima fecha del dataset: {max_order_date.date()}")
print(f"Ultimo mes cerrado usado: {last_complete_month:%Y-%m}")


## 1. Bootstrap del AOV

El AOV que reporta Q1 es un promedio mensual. Un intervalo de confianza construido con la formula cerrada (media +/- 1.96 * sigma / sqrt(n)) asume normalidad que el ticket rara vez cumple: la cola derecha la estiran los outliers de baskets caros. El bootstrap no parametrico evita ese supuesto: reemuestreamos con reemplazo las ordenes del ultimo mes cerrado y recomputamos el AOV mil veces. El intervalo empirico al 95% sale de los percentiles 2.5 y 97.5 de esa distribucion.

Se consulta el detalle por orden (no por linea) para que el AOV sea consistente con el del README. El ultimo mes cerrado es el mes anterior al maximo disponible en el dataset; si el dataset llega a abril parcial, se usa marzo.


In [ ]:
AOV_QUERY = """
SELECT
  oi.order_id,
  SUM(oi.sale_price) AS order_value
FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
WHERE oi.status NOT IN ('Cancelled', 'Returned')
  AND DATE_TRUNC(DATE(oi.created_at), MONTH) = @last_complete_month
GROUP BY oi.order_id
"""

orders = client.query(AOV_QUERY, job_config=query_params).to_dataframe(create_bqstorage_client=False)
print(f"Mes auditado: {last_complete_month:%Y-%m}")
print(f"Ordenes del ultimo mes cerrado: {len(orders):,}")
print(f"AOV observado: ${orders['order_value'].mean():.2f}")


In [ ]:
N_BOOTSTRAP = 1000
sample = orders["order_value"].to_numpy()

# Resampling con reemplazo; cada replica tiene el mismo tamano que la muestra original.
indices = rng.integers(0, len(sample), size=(N_BOOTSTRAP, len(sample)))
boot_aovs = sample[indices].mean(axis=1)

ci_lo, ci_hi = np.percentile(boot_aovs, [2.5, 97.5])
aov_point = sample.mean()

print(f"AOV observado         : ${aov_point:.2f}")
print(f"IC bootstrap 95% (1000 replicas): [${ci_lo:.2f}, ${ci_hi:.2f}]")
print(f"Ancho del intervalo   : ${ci_hi - ci_lo:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_aovs, bins=40, color="#1a73e8", alpha=0.85, edgecolor="white")
ax.axvline(aov_point, color="#202124", linestyle="-", linewidth=1.5, label=f"AOV observado ${aov_point:.2f}")
ax.axvline(ci_lo, color="#d93025", linestyle="--", linewidth=1.2, label=f"IC 2.5% ${ci_lo:.2f}")
ax.axvline(ci_hi, color="#d93025", linestyle="--", linewidth=1.2, label=f"IC 97.5% ${ci_hi:.2f}")
ax.set_title("Distribucion bootstrap del AOV (ultimo mes cerrado)")
ax.set_xlabel("AOV ($)")
ax.set_ylabel("Replicas")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

**Lectura.** Si el intervalo queda en el orden de +/-2-3 dolares, el AOV reportado es estable y las decisiones que lo usan como entrada (threshold de shipping gratis, targets de ticket promedio) se mueven sobre terreno razonable. Un intervalo mucho mas ancho implicaria que un solo mes no alcanza y conviene promediar sobre tres.


## 2. Test de significancia sobre retencion M1

El test compara la retencion M1 agregada de cohortes tempranas contra cohortes recientes, pero solo con cohortes que ya pueden observar su mes 1 completo. Si el ultimo mes cerrado es marzo 2026, la cohorte mas reciente que entra al test M1 es febrero 2026; marzo todavia no tiene abril completo y no se trata como cero.


In [ ]:
COHORT_QUERY = """
WITH valid_items AS (
  SELECT user_id, DATE_TRUNC(DATE(created_at), MONTH) AS order_month
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
),
first_order AS (
  SELECT user_id, MIN(order_month) AS cohort_month
  FROM valid_items
  GROUP BY user_id
),
cohort_size AS (
  SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_users
  FROM first_order
  GROUP BY cohort_month
),
m1_retained AS (
  SELECT
    f.cohort_month,
    COUNT(DISTINCT f.user_id) AS retained_m1
  FROM first_order f
  JOIN valid_items v
    ON v.user_id = f.user_id
   AND DATE_DIFF(v.order_month, f.cohort_month, MONTH) = 1
  GROUP BY f.cohort_month
)
SELECT
  cs.cohort_month,
  cs.cohort_users,
  COALESCE(mr.retained_m1, 0) AS retained_m1
FROM cohort_size cs
LEFT JOIN m1_retained mr USING (cohort_month)
WHERE cs.cohort_users >= 50
  AND DATE_ADD(cs.cohort_month, INTERVAL 1 MONTH) <= @last_complete_month
ORDER BY cs.cohort_month
"""

cohorts = client.query(COHORT_QUERY, job_config=query_params).to_dataframe(create_bqstorage_client=False)
cohorts["cohort_month"] = pd.to_datetime(cohorts["cohort_month"])
cohorts["m1_rate"] = cohorts["retained_m1"] / cohorts["cohort_users"]
print(f"Cohortes maduras analizadas: {len(cohorts)}")
print(f"Rango de cohortes: {cohorts['cohort_month'].min():%Y-%m} a {cohorts['cohort_month'].max():%Y-%m}")
cohorts.tail(6)


In [ ]:
median_date = cohorts["cohort_month"].median()
first_half = cohorts[cohorts["cohort_month"] <  median_date]
second_half = cohorts[cohorts["cohort_month"] >= median_date]

n1, r1 = first_half["cohort_users"].sum(),  first_half["retained_m1"].sum()
n2, r2 = second_half["cohort_users"].sum(), second_half["retained_m1"].sum()
p1, p2 = r1 / n1, r2 / n2

# Two-proportion z-test (pooled)
p_pool = (r1 + r2) / (n1 + n2)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z = (p2 - p1) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

# IC 95% para la diferencia de proporciones (Wald)
se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
diff = p2 - p1
ci_diff = (diff - 1.96*se_diff, diff + 1.96*se_diff)

print(f"Primera mitad  ({len(first_half)} cohortes): n={n1:>7,}  M1={p1*100:.2f}%")
print(f"Segunda mitad  ({len(second_half)} cohortes): n={n2:>7,}  M1={p2*100:.2f}%")
print(f"Diferencia                     : {diff*100:+.2f} pp")
print(f"IC 95% de la diferencia        : [{ci_diff[0]*100:+.2f}, {ci_diff[1]*100:+.2f}] pp")
print(f"z={z:.3f}   p-value={p_value:.3e}")
print(f"Significativo al 5%: {'si' if p_value < 0.05 else 'no'}")

**Lectura.** Un p-value chico (< 0.001) con un IC de la diferencia que no cruza cero confirma que las cohortes recientes retienen mejor en M1. La magnitud importa mas que el p-value: una mejora de ~2 puntos porcentuales en M1 es material porque se aplica sobre miles de clientes nuevos.


## 3. Heatmap de cohortes (visual, coherente con la tabla del README)

El heatmap reconstruye la tabla de retencion con las ultimas 12 cohortes de al menos 100 clientes. Solo pinta celdas observables hasta el ultimo mes cerrado; los meses futuros o incompletos quedan vacios.


In [ ]:
TRIANGLE_QUERY = """
WITH valid_items AS (
  SELECT user_id, DATE_TRUNC(DATE(created_at), MONTH) AS order_month
  FROM `bigquery-public-data.thelook_ecommerce.order_items`
  WHERE status NOT IN ('Cancelled', 'Returned')
),
first_order AS (
  SELECT user_id, MIN(order_month) AS cohort_month
  FROM valid_items GROUP BY user_id
),
cohort_size AS (
  SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_users
  FROM first_order GROUP BY cohort_month
),
activity AS (
  SELECT
    f.cohort_month,
    DATE_DIFF(v.order_month, f.cohort_month, MONTH) AS months_since,
    COUNT(DISTINCT v.user_id) AS retained
  FROM first_order f
  JOIN valid_items v USING (user_id)
  WHERE v.order_month >= f.cohort_month
    AND v.order_month <= @last_complete_month
    AND f.cohort_month <= @last_complete_month
  GROUP BY f.cohort_month, months_since
)
SELECT
  a.cohort_month,
  a.months_since,
  ROUND(a.retained / cs.cohort_users * 100, 2) AS retention_pct
FROM activity a
JOIN cohort_size cs USING (cohort_month)
WHERE a.months_since BETWEEN 0 AND 12
  AND cs.cohort_users >= 100
ORDER BY a.cohort_month, a.months_since
"""

triangle = client.query(TRIANGLE_QUERY, job_config=query_params).to_dataframe(create_bqstorage_client=False)
pivot = triangle.pivot(index="cohort_month", columns="months_since", values="retention_pct")
pivot = pivot.tail(12)
pivot.index = pd.to_datetime(pivot.index).strftime("%Y-%m")
pivot


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".1f",
    cmap="Greens",
    cbar_kws={"label": "% retained"},
    linewidths=0.3,
    linecolor="white",
    ax=ax,
)
ax.set_title("Cohort retention - last 12 observable cohorts")
ax.set_xlabel("Months since first purchase")
ax.set_ylabel("Cohorte")
fig.tight_layout()
heatmap_path = DASHBOARD_DIR / "validation_retention_heatmap.png"
fig.savefig(heatmap_path, dpi=160, bbox_inches="tight")
print(f"Heatmap exportado: {heatmap_path.relative_to(project_root)}")
plt.show()


## Cierre

La notebook no reemplaza las queries del proyecto; las audita desde otro angulo. El punto es detectar errores de grano, meses parciales y claims de retencion que podrian sonar bien en el README pero no aguantar una revision tecnica.
